# Reading in a short story as text sample into Python

In [1]:
with open("data/a-few-good-men.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    print("Total number of characters: ", len(raw_text))
    print("\n", raw_text[300:500])

Total number of characters:  162765

 able 
 sounds of night in the woods. We also HEAR, very, very 
 faintly, a slow, deliberate drum cadence. And as this starts, 
 we begin to MOVE SLOWLY UP THE TOWER, more becomes visible 
 now:... the


# Tokenizing Text

In [2]:
import re

preprocessed = re.findall(r"\w+|[^\w\s]", raw_text)
print("Number of Tokens: ", len(preprocessed))
print(preprocessed[:20])

Number of Tokens:  35834
['A', 'FEW', 'GOOD', 'MEN', 'Written', 'by', 'Aaron', 'Sorkin', 'Revised', 'Third', 'Draft', 'July', '15', ',', '1991', 'FADE', 'IN', ':', 'EXT', '.']


# Converting tokens into token IDs

In [3]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print("Vocabulary size: ", vocab_size)

Vocabulary size:  3714


## Creating a vocabulary

In [4]:
vocab = {token : i for i, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

('!', 0)
('"', 1)
('#', 2)
('&', 3)
("'", 4)
('(', 5)
(')', 6)
(',', 7)
('-', 8)
('.', 9)
('/', 10)
('00', 11)
('0600', 12)
('1', 13)
('1000', 14)
('14', 15)
('14th', 16)
('15', 17)
('16', 18)
('17', 19)
('18th', 20)


# Simple text tokenizer

In [5]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self, text):
        preprocessed = re.findall(r"\w+|[^\w\s]", text)
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        return " ".join([self.int_to_str[i] for i in ids])

tokenizer = SimpleTokenizer(vocab)
text = """Please the Court, is this dialogue
        relevant to anything in particular?"""

print("Text: ", text)
ids = tokenizer.encode(text)
print("\nEncoded text: ", ids)

print("\nDecoded text: ", tokenizer.decode(ids))

Text:  Please the Court, is this dialogue
        relevant to anything in particular?

Encoded text:  [695, 3415, 251, 7, 2195, 3431, 1598, 2914, 3453, 1078, 2142, 2661, 59]

Decoded text:  Please the Court , is this dialogue relevant to anything in particular ?


## Adding special context token

In [6]:
print("\nSize of vocab before special context token: ", len(vocab))
all_tokens = sorted(set(preprocessed))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token : i for i, token in enumerate(all_tokens)}
print("\nSize of vocab after special context token: ", len(vocab))

print("\nlast 5 elements of vocab: ")
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)


Size of vocab before special context token:  3714

Size of vocab after special context token:  3716

last 5 elements of vocab: 
('yourself', 3711)
('zero', 3712)
('zippity', 3713)
('<|endoftext|>', 3714)
('<|unk|>', 3715)


## Tokenizer that handles unknown words

In [7]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items() }

    def encode(self, text):
        preprocessed = re.findall(r"\w+|[^\w\s]", text)
        preprocessed = [s if s in self.str_to_int else "<|unk|>" for s in preprocessed]
        return [self.str_to_int[s] for s in preprocessed]

    def decode(self, ids):
        decode_text = " ". join([self.int_to_str[i] for i in ids])
        return re.sub(r"\s+([,.?!])", r"\1", decode_text)

tokenizer = SimpleTokenizerV2(vocab)
text1 = "Did you wear that uniform on the plane? The astronaut smiled."
text2 = "Did you wear that uniform on the plane? Quantum drift begins."
text = " <|endoftext|> ".join((text1, text2))

print("Text: ", text)
ids = tokenizer.encode(text)
print("\nEncoded text: ", ids)

print("\nDecoded text: ", tokenizer.decode(ids))




Text:  Did you wear that uniform on the plane? The astronaut smiled. <|endoftext|> Did you wear that uniform on the plane? Quantum drift begins.

Encoded text:  [281, 3707, 3613, 3414, 3538, 2591, 3415, 2724, 59, 876, 3715, 3715, 9, 3715, 3715, 3715, 3715, 3715, 281, 3707, 3613, 3414, 3538, 2591, 3415, 2724, 59, 3715, 3715, 1206, 9]

Decoded text:  Did you wear that uniform on the plane? The <|unk|> <|unk|>. <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> Did you wear that uniform on the plane? <|unk|> <|unk|> begins.


## Byte pair encoding

In [8]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
text = (
"Hello, do you like tea? <|endoftext|> In the sunlit terraces"
"of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print("Text: ", text)
print("\nEncoded text: ",integers)
print("\nDecoded text: ",tokenizer.decode(integers))

Text:  Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.

Encoded text:  [15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]

Decoded text:  Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


# Data sampling with a sliding window

In [9]:
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

48930


One of the easiest and most intuitive ways to create the input–target pairs for the next-
word prediction task is to create two variables, x and y , where x contains the input
tokens and y contains the targets, which are the inputs shifted by 1:

In [10]:
enc_sample = enc_text[400:]

context_size = 10
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [319, 262, 466, 967, 34952, 290, 4962, 340, 220, 198]
y:      [262, 466, 967, 34952, 290, 4962, 340, 220, 198, 6364]



By processing the inputs along with the targets, which are the inputs shifted by one
position, we can create the next-word prediction tasks (see figure 2.12), as follows:

In [11]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

[319] ----> 262
[319, 262] ----> 466
[319, 262, 466] ----> 967
[319, 262, 466, 967] ----> 34952
[319, 262, 466, 967, 34952] ----> 290
[319, 262, 466, 967, 34952, 290] ----> 4962
[319, 262, 466, 967, 34952, 290, 4962] ----> 340
[319, 262, 466, 967, 34952, 290, 4962, 340] ----> 220
[319, 262, 466, 967, 34952, 290, 4962, 340, 220] ----> 198
[319, 262, 466, 967, 34952, 290, 4962, 340, 220, 198] ----> 6364


Everything left of the arrow (---->) refers to the input an LLM would receive, and
the token ID on the right side of the arrow represents the target token ID that the
LLM is supposed to predict. Let’s repeat the previous code but convert the token IDs
into text:

In [12]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 on ---->  the
 on the ---->  do
 on the do ----> ork
 on the doork ----> nob
 on the doorknob ---->  and
 on the doorknob and ---->  turns
 on the doorknob and turns ---->  it
 on the doorknob and turns it ---->  
 on the doorknob and turns it  ----> 

 on the doorknob and turns it 
 ---->  slowly


We’ve now created the input–target pairs that we can use for LLM training.

# Dataset for batched inputs and targets

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids)-max_length, stride):
            input_chunk = token_ids[i: i + max_length]
            target_chunk = token_ids[i+1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self):
        return len(self.input_ids) # Returns the total number of rows in the dataset
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx] # Returns a single row from the dataset


# Data loader to generate batches with input-with pairs

In [14]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last, # drop_last=True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training.
        num_workers=num_workers # The number of CPU processes to use for preprocessing
    )

    return dataloader

# Creating token embeddings

In [15]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[  317,   376,  6217, 21090],
        [41597,   628, 22503,   416],
        [  628, 12139,   311,   967],
        [  259,   628, 31492, 10467],
        [13650,   628,  2901,  1315],
        [   11, 10249,   628,   220],
        [  628,   376, 19266,  3268],
        [   25,   628, 27489,    13]])

Inputs shape:
 torch.Size([8, 4])


As we can see, the token ID tensor is 8 × 4 dimensional, meaning that the data batch
consists of eight text samples with four tokens each.

In [16]:
vocab_size = 48930
output_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


Using token_embedding_layer we sample data from the data loader,
we embed each token in each batch into a 256-dimensional vector. If we have a batch
size of 8 with four tokens each, the result will be an 8 × 4 × 256 tensor.

In [17]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print(pos_embeddings.shape)

torch.Size([4, 256])


The input to the pos_embeddings is usually a placeholder vector torch.arange(con-
text_length), which contains a sequence of numbers 0, 1, ..., up to the maximum
input length –1. The context_length is a variable that represents the supported input
size of the LLM. Here, we choose it similar to the maximum length of the input text.

In [18]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


As we can see, the positional embedding tensor consists of four 256-dimensional vec-
tors. We can now add these directly to the token embeddings, where PyTorch will add
the 4 × 256–dimensional pos_embeddings tensor to each 4 × 256–dimensional token
embedding tensor in each of the eight batches: